# GRPO Training with NVIDIA NeMo RL on Amazon SageMaker Training Jobs

This notebook demonstrates how to run GRPO (Group Relative Policy Optimization) using [NVIDIA NeMo RL](https://github.com/NVIDIA-NeMo/RL) on SageMaker.

NeMo RL is a Ray-based post-training library. GRPO is a reinforcement learning algorithm that:
- Generates multiple responses per prompt
- Evaluates them with a reward function (e.g., math correctness)
- Uses group-relative advantages to update the policy

We use a [custom Ray launcher](https://github.com/aws-samples/sample-ray-on-amazon-sagemaker-training-jobs) to bootstrap the Ray cluster on SageMaker training instances, then NeMo RL connects to it via `ray.init(address="auto")`.

This example trains **Qwen2.5-1.5B** on **OpenMathInstruct-2** with math reward verification.

## Prerequisites
### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B"

---
## NeMo RL GRPO Configuration

GRPO requires:
- **Policy config**: model, optimizer, generation backend (vLLM)
- **GRPO config**: number of generations per prompt, advantage estimation
- **Loss function**: KL penalty, ratio clipping
- **Data config**: dataset with math problems
- **Environment config**: reward function (math verification)

This uses `OpenMathInstruct-2` from HuggingFace Hub with `hf_math_verify` reward.

In [ ]:
%%bash

cat > ./grpo.yaml <<'EOF'
# NeMo RL GRPO config — Qwen2.5-1.5B with OpenMathInstruct-2 on SageMaker
# Compatible with NeMo RL v0.5.0 (NGC container nvcr.io/nvidia/nemo-rl:v0.5.0)
# Based on: examples/configs/grpo_math_1B.yaml

grpo:
  num_prompts_per_step: 32
  num_generations_per_prompt: 16
  max_rollout_turns: 1
  max_num_epochs: 1
  max_num_steps: 100
  normalize_rewards: true
  use_leave_one_out_baseline: true
  val_period: 10
  val_at_start: false
  overlong_filtering: false
  max_val_samples: 256
  val_batch_size: 256
  seed: 42
  use_dynamic_sampling: false
  dynamic_sampling_max_gen_batches: 10
  batch_multiplier: 1
  reward_shaping:
    enabled: false
    overlong_buffer_length: 128
    overlong_buffer_penalty: 1
    max_response_length: ${policy.max_total_sequence_length}
  reward_scaling:
    enabled: false
    source_min: 0.0
    source_max: 1.0
    target_min: 0.0
    target_max: 1.0
  async_grpo:
    enabled: false
    max_trajectory_age_steps: 1
    in_flight_weight_updates: false
    recompute_kv_cache_after_weight_updates: false

loss_fn:
  reference_policy_kl_penalty: 0.01
  reference_policy_kl_type: k3
  kl_input_clamp_value: 20.0
  kl_output_clamp_value: 10.0
  ratio_clip_min: 0.2
  ratio_clip_max: 0.2
  ratio_clip_c: null
  use_on_policy_kl_approximation: false
  use_importance_sampling_correction: false
  truncated_importance_sampling_ratio: null
  sequence_level_importance_ratios: false
  token_level_loss: true
  force_on_policy_ratio: false

checkpointing:
  enabled: true
  checkpoint_dir: /opt/ml/checkpoints/
  metric_name: "val:accuracy"
  higher_is_better: true
  keep_top_k: 3
  save_period: 10
  checkpoint_must_save_by: null
  model_save_format: safetensors
  save_consolidated: false

policy:
  model_name: Qwen/Qwen2.5-1.5B
  tokenizer:
    name: ${policy.model_name}
    chat_template_kwargs: null
  hf_config_overrides: {}
  train_global_batch_size: 512
  train_micro_batch_size: 4
  generation_batch_size: 32
  logprob_batch_size: ${policy.train_micro_batch_size}
  max_total_sequence_length: 512
  precision: bfloat16
  logprob_chunk_size: null
  offload_optimizer_for_logprob: false

  dtensor_cfg:
    _v2: true
    enabled: true
    cpu_offload: false
    sequence_parallel: false
    activation_checkpointing: false
    tensor_parallel_size: 1
    context_parallel_size: 1
    custom_parallel_plan: null

  megatron_cfg:
    enabled: false

  dynamic_batching:
    enabled: false
    train_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.train_micro_batch_size}}
    logprob_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.logprob_batch_size}}
    sequence_length_round: 64

  sequence_packing:
    enabled: true
    train_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.train_micro_batch_size}}
    logprob_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.logprob_batch_size}}
    algorithm: modified_first_fit_decreasing
    sequence_length_round: 64

  make_sequence_length_divisible_by: ${policy.dtensor_cfg.tensor_parallel_size}
  max_grad_norm: 1.0

  optimizer:
    name: torch.optim.AdamW
    kwargs:
      lr: 5.0e-6
      weight_decay: 0.01
      betas: [0.9, 0.999]
      eps: 1e-8
      foreach: false
      fused: false

  scheduler:
    - name: torch.optim.lr_scheduler.LinearLR
      kwargs:
        start_factor: 0.1
        end_factor: 1.0
        total_iters: 50
    - name: torch.optim.lr_scheduler.ConstantLR
      kwargs:
        factor: 1.0
        total_iters: 10000000000
    - milestones: [50]

  generation:
    backend: vllm
    max_new_tokens: ${policy.max_total_sequence_length}
    temperature: 1.0
    top_p: 1.0
    top_k: null
    stop_token_ids: null
    stop_strings: null
    vllm_cfg:
      async_engine: false
      precision: ${policy.precision}
      kv_cache_dtype: auto
      tensor_parallel_size: 1
      pipeline_parallel_size: 1
      expert_parallel_size: 1
      gpu_memory_utilization: 0.6
      max_model_len: ${policy.max_total_sequence_length}
      enforce_eager: false
      use_deep_gemm: false
      num_last_layers_in_bf16: 0
      num_first_layers_in_bf16: 0
      enable_vllm_metrics_logger: true
      vllm_metrics_logger_interval: 0.5
    vllm_kwargs: {}
    colocated:
      enabled: true
      resources:
        gpus_per_node: null
        num_nodes: null

# v0.5.0 flat data config format
data:
  max_input_seq_length: ${policy.max_total_sequence_length}
  prompt_file: "examples/prompts/cot.txt"
  system_prompt_file: null
  shuffle: true
  num_workers: 1
  processor: "math_hf_data_processor"
  env_name: "math"
  dataset_name: "OpenMathInstruct-2"

env:
  math:
    num_workers: 8
    math_verify_impl: hf_math_verify

logger:
  log_dir: /opt/ml/output/data/logs
  num_val_samples_to_print: 0
  wandb_enabled: false
  tensorboard_enabled: true
  mlflow_enabled: false
  swanlab_enabled: false
  monitor_gpus: true
  tensorboard:
    log_dir: /opt/ml/output/data/tb_logs
  gpu_monitoring:
    collection_interval: 10
    flush_interval: 10

cluster:
  gpus_per_node: 8 # Match num of GPUs for the selected instance type
  num_nodes: 1 # Match num of nodes for SageMaker Training jobs
EOF

Upload the config to S3.

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/nemo-rl"
else:
    input_path = "nemo-rl"

train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/grpo.yaml"

s3_client.upload_file("grpo.yaml", bucket_name, f"{input_path}/config/grpo.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

---
## Train Model with GRPO

NeMo RL uses Ray for distributed execution. We use the [Ray launcher for SageMaker](https://github.com/aws-samples/sample-ray-on-amazon-sagemaker-training-jobs) to bootstrap the Ray cluster, then NeMo RL connects to it.

The launcher is included in the `scripts/` directory alongside the training script. The entry point is `launcher.py --entrypoint train_grpo.py`.

#### Get PyTorch image_uri

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session

sts = boto3.client("sts")

sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
instance_type = "ml.p5.48xlarge"
instance_count = 1

instance_type

In [ ]:
account_id = sts.get_caller_identity()["Account"]
region = sagemaker_session.boto_session.region_name
repo_name = "nemo-rl-sagemaker"
tag = "latest"

image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:{tag}"

image_uri

#### Define and launch ModelTrainer

Note: We do NOT use `Torchrun` here. The entry point is `launcher.py` which bootstraps Ray and then loads `train_grpo.py` via `--entrypoint`.

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    RemoteDebugConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

args = [
    "--entrypoint",
    "train_grpo.py",
]

# Define the script to be run
# 1. setup.sh clones NeMo RL source (pip only installs top-level module)
# 2. PYTHONPATH prepends the full source so all subpackages are importable
# 3. launcher.py bootstraps Ray cluster and loads train_grpo.py
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"source setup.sh && python launcher.py {' '.join(args)}",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=0,
)

# Define Training Job Name
job_name = f"nemo-rl-{model_id.split('/')[-1].replace('.', '-')}-grpo"

# Define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer — no Torchrun, Ray handles distribution
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=16000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
    environment={
        "NEMO_RL_PY_EXECUTABLES_SYSTEM": "1",  # Use system Python, not uv
        "NCCL_P2P_DISABLE": "1",
        "NCCL_IB_DISABLE": "0",
        "NCCL_TIMEOUT": "1800",
        # "launch_prometheus": "true", # enable for local prometheus
        # "RAY_PROMETHEUS_HOST": "<PROMETHEUS_HOST>", # Visit https://github.com/aws-samples/sample-ray-on-amazon-sagemaker-training-jobs
        # "RAY_GRAFANA_HOST": "<GRAFANA_HOST>", # Visit https://github.com/aws-samples/sample-ray-on-amazon-sagemaker-training-jobs
        # "RAY_PROMETHEUS_NAME": "prometheus", # Visit https://github.com/aws-samples/sample-ray-on-amazon-sagemaker-training-jobs
    },
).with_remote_debug_config(RemoteDebugConfig(enable_remote_debug=True))

In [ ]:
from sagemaker.train.configs import InputData

config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path,
)

data = [config_input]
data

In [ ]:
model_trainer.train(input_data_config=data, wait=False)